# N1 — Data Preparation & Feature Engineering

Loads the raw Compustat panel, applies financial filters, engineers the full Geertsema & Lu (2023) ratio set, assigns FF49 industry codes, and saves `panel_clean.parquet`.

In [1]:
# Cell 0 — install dependencies
import subprocess
packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "pyarrow",    # parquet read/write
    "fastparquet", # parquet fallback
    "scikit-learn", # used in later notebooks
    "torch",       # FinBERT in N3
    "transformers", # FinBERT in N3
    "faiss-cpu",   # fast kNN in N6/N7
]
for pkg in packages:
    subprocess.run(["pip", "install", pkg, "-q"], check=True)
print("All packages installed.")

All packages installed.


In [2]:
# Cell 1 — imports & config
import sys; sys.path.insert(0, '..')
from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print("Config loaded.")
print(f"  COMPUSTAT_FILE : {COMPUSTAT_FILE}")
print(f"  PANEL_CLEAN    : {PANEL_CLEAN}")
print(f"  YEARS          : {YEARS}")


Config loaded.
  COMPUSTAT_FILE : /work/Repo/notebooks/../data/raw/financial/perfect_compustat_pull_v2.csv
  PANEL_CLEAN    : /work/Repo/notebooks/../data/processed/panel_clean.parquet
  YEARS          : [2020, 2021, 2022, 2023, 2024]


In [3]:
# Cell 2 — declare I/O
INPUTS  = [COMPUSTAT_FILE]
OUTPUTS = [PANEL_CLEAN]
# PURPOSE : Load Compustat panel, apply financial filters, engineer ratios, produce panel_clean.parquet
# RUNTIME : ~5 min
# DEPENDS : perfect_compustat_pull_v2.csv


## 1. Data Exploration

Quick sanity check on the raw file before any transformations.

In [4]:
# Cell 3 — raw file exploration
try:
    df_raw = pd.read_csv(COMPUSTAT_FILE, nrows=0)
    df_raw.columns = [c.lower() for c in df_raw.columns]

    # Full load for exploration
    df_raw = pd.read_csv(COMPUSTAT_FILE)
    df_raw.columns = [c.lower() for c in df_raw.columns]

    print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]}")
    if 'gvkey' in df_raw.columns:
        print(f"Unique firms (GVKEY): {df_raw['gvkey'].nunique():,}")
    if 'tic' in df_raw.columns:
        print(f"Unique firms (Ticker): {df_raw['tic'].nunique():,}")

    # Column check against expected schema
    expected_cols = [
        'tic', 'sic', 'fyear', 'prcc_f', 'csho', 'sale', 'at', 'seq', 'dlc', 'dltt',
        'act', 'che', 'rect', 'invt', 'ppent', 'lt', 'lct', 'ap', 'ceq', 'pstk',
        'mib', 'icapt', 'txditc', 'cogs', 'xsga', 'xrd', 'dp', 'xint', 'spi',
        'nopi', 'pi', 'txt', 'mii', 'ib', 'ibcom', 'ni', 'ebitda', 'ebit',
        'oancf', 'capx', 'dv'
    ]
    missing_cols = [c for c in expected_cols if c not in df_raw.columns]
    print(f"\nRequired columns: {'all present' if not missing_cols else f'MISSING: {missing_cols}'}")

    # Missing data on key variables
    print("\nMissing data (key variables):")
    for col in [c for c in ['sale', 'at', 'seq', 'ni', 'xrd', 'dp', 'xint', 'rect', 'invt'] if c in df_raw.columns]:
        print(f"  {col.upper():<8}: {df_raw[col].isna().mean()*100:.1f}%")

    # Financial / utility sector counts
    if {'sic', 'prcc_f', 'csho', 'lt', 'at'}.issubset(df_raw.columns):
        df_raw['sic_n'] = pd.to_numeric(df_raw['sic'], errors='coerce')
        df_raw['market_cap_'] = df_raw['csho'] * df_raw['prcc_f']
        fin = df_raw[df_raw['sic_n'].between(6000, 6999)]
        non_fin = df_raw[~df_raw['sic_n'].between(6000, 6999)]
        print(f"\nFinancials (SIC 6xxx): {len(fin):,} firm-years")
        print(f"Utilities  (SIC 49xx): {df_raw['sic_n'].between(4900, 4999).sum():,} firm-years")

    if 'xrd' in df_raw.columns:
        print(f"\nMissing R&D: {df_raw['xrd'].isna().sum():,} firm-years → filled with 0 during feature engineering")

    if 'prcc_f' in df_raw.columns:
        df_raw['market_cap_'] = df_raw['csho'] * df_raw['prcc_f']
        excl = ((df_raw['prcc_f'] < MIN_PRICE) | (df_raw['market_cap_'] < MIN_MKTCAP)).sum()
        print(f"Penny/micro-cap exclusions (would be): {excl:,} firm-years")

except FileNotFoundError:
    print(f"File not found: {COMPUSTAT_FILE}")
    print("Place the raw file at the path above and re-run.")
except Exception as e:
    print(f"Error: {e}")
    raise


Rows: 169,724 | Columns: 50
Unique firms (GVKEY): 21,674
Unique firms (Ticker): 21,654

Required columns: all present

Missing data (key variables):
  SALE    : 35.5%
  AT      : 26.8%
  SEQ     : 26.8%
  NI      : 35.5%
  XRD     : 70.3%
  DP      : 30.3%
  XINT    : 35.2%
  RECT    : 36.0%
  INVT    : 33.5%

Financials (SIC 6xxx): 79,124 firm-years
Utilities  (SIC 49xx): 4,406 firm-years

Missing R&D: 119,303 firm-years → filled with 0 during feature engineering
Penny/micro-cap exclusions (would be): 46,449 firm-years


## 2. Data Dictionary

Mapping of all raw Compustat items and engineered features used in the pipeline.

In [5]:
# Cell 4 — data dictionary (documentation only; not used in pipeline)
data_dictionary = {
    # Identifiers & Market Data (Raw)
    'gvkey'  : 'Global Company Key (Compustat internal unique identifier)',
    'tic'    : 'Ticker Symbol',
    'conm'   : 'Company Name',
    'sic'    : 'Standard Industrial Classification Code',
    'fyear'  : 'Fiscal Year',
    'prcc_f' : 'Closing Stock Price at Fiscal Year-End',
    'csho'   : 'Common Shares Outstanding (millions)',
    'curcd'  : "Currency Code — filtered strictly to 'USD'",

    # Income Statement
    'sale'   : 'Net Sales / Total Revenue',
    'cogs'   : 'Cost of Goods Sold',
    'xsga'   : 'Selling, General and Administrative Expense',
    'xrd'    : 'Research and Development Expense',
    'dp'     : 'Depreciation and Amortization',
    'xint'   : 'Interest and Related Expense',
    'spi'    : 'Special Items',
    'nopi'   : 'Nonoperating Income / Expense',
    'pi'     : 'Pretax Income',
    'txt'    : 'Income Taxes Total',
    'mii'    : 'Noncontrolling Interest (Income Account)',
    'ib'     : 'Income Before Extraordinary Items',
    'ibcom'  : 'Income Before Extraordinary Items Available for Common',
    'ni'     : 'Net Income',
    'ebitda' : 'Earnings Before Interest, Taxes, Depreciation, and Amortization',
    'ebit'   : 'Earnings Before Interest and Taxes',

    # Balance Sheet
    'at'     : 'Total Assets',
    'act'    : 'Total Current Assets',
    'che'    : 'Cash and Short-Term Investments',
    'rect'   : 'Receivables - Total',
    'invt'   : 'Inventories - Total',
    'ppent'  : 'Property, Plant and Equipment (Net)',
    'lt'     : 'Total Liabilities',
    'lct'    : 'Total Current Liabilities',
    'ap'     : 'Accounts Payable - Trade',
    'dlc'    : 'Debt in Current Liabilities (Short-Term Debt)',
    'dltt'   : 'Long-Term Debt',
    'seq'    : "Total Shareholders' Equity (Parent)",
    'ceq'    : 'Common/Ordinary Equity',
    'pstk'   : 'Preferred Stock',
    'mib'    : 'Noncontrolling Interests (Balance Sheet)',
    'icapt'  : 'Invested Capital',
    'txditc' : 'Deferred Taxes and Investment Tax Credit',

    # Cash Flow & Dividends
    'oancf'  : 'Operating Activities - Net Cash Flow',
    'capx'   : 'Capital Expenditures',
    'dv'     : 'Cash Dividends - Total',

    # Engineered — Target Multiples (N4)
    'market_cap' : 'Market Capitalization = csho * prcc_f',
    'ev'         : 'Enterprise Value = Market Cap + dltt + dlc - che',
    'ln_m2b'     : '[TARGET] ln(Market Cap / seq)',
    'ln_v2a'     : '[TARGET] ln(EV / Total Assets)',
    'ln_v2s'     : '[TARGET] ln(EV / Sales)',
}

print(f"Dictionary contains {len(data_dictionary)} entries.")
print("\nTarget multiples:")
for k, v in {k: v for k, v in data_dictionary.items() if '[TARGET]' in v}.items():
    print(f"  {k:<10} : {v}")


Dictionary contains 49 entries.

Target multiples:
  ln_m2b     : [TARGET] ln(Market Cap / seq)
  ln_v2a     : [TARGET] ln(EV / Total Assets)
  ln_v2s     : [TARGET] ln(EV / Sales)


## 3. Data Validation

Check for duplicates, currency issues, data types, and the R&D null trap before the pipeline runs.

In [6]:
# Cell 5 — data validation (run before pipeline)
df_check = pd.read_csv(COMPUSTAT_FILE)
df_check.columns = [c.lower() for c in df_check.columns]

# 1. Duplicates
if 'gvkey' in df_check.columns and 'fyear' in df_check.columns:
    n_dupes = df_check.duplicated(subset=['gvkey', 'fyear'], keep=False).sum()
    print(f"Duplicates on (gvkey, fyear): {n_dupes:,}" +
          (" — will keep='last' after sorting by datadate" if n_dupes else " — none found"))

# 2. Currency
if 'curcd' in df_check.columns:
    print("\nCurrency breakdown:")
    for curr, count in df_check['curcd'].value_counts(dropna=False).head(8).items():
        flag = " ← KEEP" if curr == 'USD' else " ← DROP"
        print(f"  {str(curr):<6}: {count:>8,}{flag}")
else:
    print("No currency column found — assuming all USD.")

# 3. Data types on numeric columns
numeric_cols = [
    'prcc_f', 'csho', 'sale', 'at', 'seq', 'dlc', 'dltt', 'act', 'che', 'rect',
    'invt', 'ppent', 'lt', 'lct', 'ap', 'ceq', 'pstk', 'mib', 'icapt', 'txditc',
    'cogs', 'xsga', 'xrd', 'dp', 'xint', 'spi', 'nopi', 'pi', 'txt', 'mii',
    'ib', 'ibcom', 'ni', 'ebitda', 'ebit', 'oancf', 'capx', 'dv'
]
non_numeric = [c for c in numeric_cols if c in df_check.columns
               and not pd.api.types.is_numeric_dtype(df_check[c])]
print(f"\nData types: {'all numeric ✓' if not non_numeric else f'NON-NUMERIC: {non_numeric}'}")

# 4. R&D null trap — NaN in Compustat means $0, NOT missing
if 'xrd' in df_check.columns:
    print(f"\nR&D (xrd) nulls: {df_check['xrd'].isna().sum():,} firm-years")
    print("  → Compustat encodes zero R&D as NaN. Will fill with 0 during feature engineering.")

del df_check


Duplicates on (gvkey, fyear): 28,712 — will keep='last' after sorting by datadate

Currency breakdown:
  USD   :  141,563 ← KEEP
  CAD   :   28,161 ← DROP

Data types: all numeric ✓

R&D (xrd) nulls: 119,303 firm-years
  → Compustat encodes zero R&D as NaN. Will fill with 0 during feature engineering.


## 4. Full Data Preparation Pipeline

Runs end-to-end: load → clean → lags → 70+ ratios → FF49 → filters → save `panel_clean.parquet`.

In [7]:
# Cell 6 — load raw file
df = pd.read_csv(COMPUSTAT_FILE)
df.columns = [c.lower() for c in df.columns]

_log = []   # attrition tracker: (step_label, n_firm_years, n_companies)
_log.append(('1. Raw file', len(df), df['gvkey'].nunique()))
print(f"Loaded: {len(df):,} rows | {df['gvkey'].nunique():,} unique GVKEYs")


Loaded: 169,724 rows | 21,674 unique GVKEYs


In [8]:
# Cell 7 — currency filter & deduplication
# Step 2: drop non-USD
if 'curcd' in df.columns:
    n_before = len(df)
    df = df[df['curcd'] == 'USD'].copy()
    print(f"USD filter: dropped {n_before - len(df):,} non-USD rows")
_log.append(('2. After USD filter', len(df), df['gvkey'].nunique()))

# Sort so keep='last' retains the most recent datadate entry per firm-year
if 'datadate' in df.columns:
    df = df.sort_values(by=['gvkey', 'fyear', 'datadate'])
else:
    df = df.sort_values(by=['gvkey', 'fyear'])

n_before = len(df)
df = df.drop_duplicates(subset=['gvkey', 'fyear'], keep='last')
print(f"Deduplication: dropped {n_before - len(df):,} duplicate firm-years")
_log.append(('3. After deduplication', len(df), df['gvkey'].nunique()))
print(f"After filter & dedup: {len(df):,} rows")


USD filter: dropped 28,161 non-USD rows
Deduplication: dropped 13,615 duplicate firm-years
After filter & dedup: 127,948 rows


In [9]:
# Cell 8 — base variable construction
# Book equity (Fama-French definition)
df['be']    = df['seq'] + df['txditc'].fillna(0) - df['pstk'].fillna(0)
df['ocf']   = df['oancf']
df['ibadj'] = df['ib']

# Ensure optional income statement items exist (fill with 0 if absent)
for safe_col in ['mii', 'nopi', 'spi', 'xrd']:
    if safe_col not in df.columns:
        df[safe_col] = 0.0

print("Base variables constructed: be, ocf, ibadj")
print(f"  be: {df['be'].notna().sum():,} non-null")


Base variables constructed: be, ocf, ibadj
  be: 91,255 non-null


In [10]:
# Cell 9 — time-series lags (t-1) and first differences
print("Calculating lags and deltas...")

def lag(df, col):
    return df.groupby('gvkey')[col].shift(1)

def delta(df, col, l_col):
    return df[col] - df[l_col]

base_vars = [
    'at', 'sale', 'seq', 'lt', 'invt', 'capx', 'ni', 'ebitda', 'cogs', 'che',
    'act', 'lct', 'rect', 'ap', 'ib', 'dp', 'ppent', 'xsga', 'icapt', 'txditc',
    'mib', 'ceq', 'be', 'dlc', 'dltt', 'pstk'
]

for col in base_vars:
    if col in df.columns:
        df[f'l_{col}'] = lag(df, col)
        df[f'd_{col}'] = delta(df, col, f'l_{col}')

print(f"Lags and deltas created for {sum(1 for c in base_vars if c in df.columns)} variables")


Calculating lags and deltas...
Lags and deltas created for 26 variables


In [11]:
# Cell 10 — Geertsema & Lu (2023) ratio engineering: Profitability & Margins
df = df.copy()
print("Engineering Geertsema & Lu ratios — (1) Profitability & Margins...")

# Returns & Profitability
df['Accrual_wr']        = (df['ocf'] - df['ib']) / df['at']
df['aftret_eq_wr']      = df['ibcom'] / df['ceq']
df['aftret_equity_wr']  = df['ib'] / df['seq']
denom_invcap = (df['l_icapt'] + df['l_txditc'].fillna(0) - df['l_mib'].fillna(0))
df['aftret_invcapx_wr'] = np.where(denom_invcap > 0,
                                    (df['ib'] + df['xint'] + df['mii'].fillna(0)) / denom_invcap,
                                    np.nan)
df['roa_wr']            = df['ebitda'] / df['at']
df['roce_wr']           = df['ebit'] / (df['dltt'].fillna(0) + df['pstk'].fillna(0)
                                         + df['dlc'].fillna(0) + df['ceq'].fillna(0))
df['roe_wr']            = np.where(df['be'] > 0, df['ib'] / df['be'], np.nan)
df['roic_an']           = (df['ebit'] - df['nopi'].fillna(0)) / (df['ceq'].fillna(0)
                                                                   + df['lt'] - df['che'].fillna(0))
df['profitability_an']  = (df['sale'] - df['cogs'] - df['xint'] - df['xsga']) / df['be']
df['pretret_earnat_wr'] = df['ebitda'] / (df['ppent'] + df['act'])
df['pretret_noa_wr']    = df['ebit'] / (df['ppent'] + df['act'] - df['lct'])

# Margins
df['gpm_wr']            = (df['sale'] - df['cogs']) / df['sale']
df['grossmargin_an']    = df['gpm_wr']
df['gprof_wr']          = (df['sale'] - df['cogs']) / df['at']
df['grossprofit_an']    = df['gprof_wr']
df['npm_wr']            = df['ib'] / df['sale']
df['opmad_wr']          = df['ebit'] / df['sale']
df['opmbd_wr']          = df['ebitda'] / df['sale']
df['ebitda2revenue_an'] = df['opmbd_wr']
df['ptpm_wr']           = (df['ebit'] - df['xint'] + df['spi'].fillna(0)
                            + df['nopi'].fillna(0)) / df['sale']
df['cfm_wr']            = (df['ib'] + df['dp']) / df['sale']

print("  ✓ Profitability & Margins complete")


Engineering Geertsema & Lu ratios — (1) Profitability & Margins...
  ✓ Profitability & Margins complete


In [12]:
# Cell 11 — Geertsema & Lu ratios: Turnover & Efficiency
df = df.copy()
print("Engineering ratios — (2) Turnover & Efficiency...")

df['assetturnover_an']  = df['sale'] / df['at']
df['at_turn_wr']        = df['assetturnover_an']
df['assetturnover2_an'] = df['sale'] / ((df['at'] + df['l_at']) / 2)
df['invturn_wr']        = np.where((df['invt'] + df['l_invt']) / 2 > 0,
                                    df['cogs'] / ((df['invt'] + df['l_invt']) / 2), np.nan)
df['rect_turn_wr']      = df['sale'] / ((df['rect'] + df['l_rect']) / 2)
df['pay_turn_wr']       = np.where((df['ap'] + df['l_ap']) / 2 > 0,
                                    (df['cogs'] + df['d_invt']) / ((df['ap'] + df['l_ap']) / 2),
                                    np.nan)
df['sales2cash_an']     = df['sale'] / df['che']
df['sales2inv_an']      = df['sale'] / df['invt']
df['sales2rec_an']      = df['sale'] / df['rect']
df['opleverage_an']     = (df['cogs'] + df['xsga']) / df['at']
df['rd_sale_wr']        = np.maximum(df['xrd'].fillna(0) / df['sale'], 0)

print("  ✓ Turnover & Efficiency complete")


Engineering ratios — (2) Turnover & Efficiency...
  ✓ Turnover & Efficiency complete


In [13]:
# Cell 12 — Geertsema & Lu ratios: Leverage & Capital Structure
df = df.copy()
print("Engineering ratios — (3) Leverage & Capital Structure...")

df['debt_assets_wr']    = df['lt'] / df['at']
df['debt_at_wr']        = (df['dltt'] + df['dlc']) / df['at']
df['de_ratio_wr']       = df['lt'] / (df['ceq'].fillna(0) + df['pstk'].fillna(0))
df['capital_ratio_wr']  = df['dltt'] / (df['dltt'] + df['ceq'].fillna(0) + df['pstk'].fillna(0))
df['debt_capital_wr']   = ((df['ap'] + df['dlc'].fillna(0) + df['dltt'].fillna(0))
                            / (df['ap'] + df['dlc'].fillna(0) + df['dltt'].fillna(0)
                               + df['ceq'].fillna(0) + df['pstk'].fillna(0)))
df['debt_ebitda_wr']    = (df['dltt'] + df['dlc']) / df['ebitda']
df['debt_invcap_wr']    = np.where(df['icapt'] > 0, df['dltt'] / df['icapt'], np.nan)
df['totdebt_invcap_wr'] = np.where(df['icapt'] > 0, (df['dltt'] + df['dlc']) / df['icapt'], np.nan)
df['curr_debt_wr']      = df['lct'] / df['lt']
df['lt_debt_wr']        = df['dltt'] / df['lt']
df['short_debt_wr']     = df['dlc'] / (df['dltt'] + df['dlc'])
df['lt_ppent_wr']       = df['lt'] / df['ppent']
df['debt2tang_an']      = (df['che'] + df['rect'] * 0.715 + df['invt'] * 0.547
                            + df['ppent'] * 0.535) / df['at']
df['debt_book_value']   = df['dltt'].fillna(0) + df['dlc'].fillna(0) + df['pstk'].fillna(0)
df['dltt_be_wr']        = np.where(df['be'] > 0, df['dltt'] / df['be'], np.nan)
df['equity_invcap_wr']  = np.where(df['icapt'] > 0, df['ceq'] / df['icapt'], np.nan)

print("  ✓ Leverage & Capital Structure complete")


Engineering ratios — (3) Leverage & Capital Structure...
  ✓ Leverage & Capital Structure complete


In [14]:
# Cell 13 — Geertsema & Lu ratios: Liquidity & Cash Flow
df = df.copy()
print("Engineering ratios — (4) Liquidity & Cash Flow...")

df['cash2assets_an']     = df['che'] / df['at']
df['cash_lt_wr']         = df['che'] / df['lt']
df['cash_debt_wr']       = df['ocf'] / df['lt']
df['cf2debt_an']         = (df['ib'] + df['dp']) / ((df['lt'] + df['l_lt']) / 2)
df['curr_ratio_wr']      = df['act'] / df['lct']
df['quick_ratio_wr']     = np.where(df['lct'] > 0, (df['act'] - df['invt']) / df['lct'], np.nan)
df['cash_ratio_wr']      = np.where(df['lct'] > 0, df['che'] / df['lct'], np.nan)
df['invt_act_wr']        = df['invt'] / df['act']
df['rect_act_wr']        = df['rect'] / df['act']
df['ocf_lct_wr']         = df['ocf'] / df['lct']
df['profit_lct_wr']      = df['ebitda'] / df['lct']
df['fcf_ocf_wr']         = np.where(df['ocf'] > 0, (df['ocf'] - df['capx']) / df['ocf'], np.nan)
df['liquid2assets_an']   = (df['che'] + 0.75 * (df['act'] - df['che'])
                             + 0.5 * (df['at'] - df['act'])) / df['l_at']
df['cash_conversion_wr'] = (
    ((df['invt']  + df['l_invt'])  / 2) / (df['cogs'] / 365) +
    ((df['rect']  + df['l_rect'])  / 2) / (df['sale'] / 365) -
    ((df['ap']    + df['l_ap'])    / 2) / (df['cogs'] / 365)
)

print("  ✓ Liquidity & Cash Flow complete")


Engineering ratios — (4) Liquidity & Cash Flow...
  ✓ Liquidity & Cash Flow complete


In [15]:
# Cell 14 — Geertsema & Lu ratios: Coverage & Rates
df = df.copy()
print("Engineering ratios — (5) Coverage & Rates...")

df['intcov_ratio_wr'] = df['ebit'] / df['xint']
df['intcov_wr']       = (df['xint'] + df['ib']) / df['xint']
df['int_debt_wr']     = df['xint'] / df['dltt']
df['int_totdebt_wr']  = df['xint'] / (df['dltt'] + df['dlc'])
df['deprn_an']        = df['dp'] / df['ppent']
df['efftax_wr']       = df['txt'] / df['pi']
df['dpr_wr']          = df['dv'] / df['ibadj']

print("  ✓ Coverage & Rates complete")


Engineering ratios — (5) Coverage & Rates...
  ✓ Coverage & Rates complete


In [16]:
# Cell 15 — Geertsema & Lu ratios: Growth & Deltas
df = df.copy()
print("Engineering ratios — (6) Growth & Deltas...")

df['assetgrowth_an']       = df['d_at'] / df['l_at']
df['bookequitygrowth_an']  = (df['be'] / df['l_be']) - 1
df['capexgrowth_an']       = df['d_capx'] / df['l_capx']
df['inventorygrowth_an']   = df['d_invt'] / df['l_invt']
df['inventorychange_an']   = df['d_invt'] / (0.5 * df['l_at'] + 0.5 * df['at'])
df['salesgrowth_an']       = (df['sale'] / df['l_sale']) - 1
df['investment_an']        = df['d_at'] / df['at']

df['chbe_an']             = (df['be'] - df['l_be']) / df['l_at']
df['chca_an']             = (df['d_act'] - df['d_che']) / df['l_at']
df['chceq_an']            = (df['ceq'] - df['l_ceq']) / df['l_ceq']
df['chcl_an']             = (df['d_lct'] - df['d_dlc'].fillna(0)) / df['l_at']
df['chcurrentratio_an']   = ((df['act'] / df['lct']) - (df['l_act'] / df['l_lct'])) / (df['l_act'] / df['l_lct'])
df['chfnl_an']            = (df['d_dltt'].fillna(0) + df['d_dlc'].fillna(0)
                              + df['d_pstk'].fillna(0)) / df['l_at']
df['chlt_an']             = (df['lt'] / df['l_lt']) - 1
df['chnccl_an']           = (df['d_lt'] - df['d_lct'] - df['d_dltt'].fillna(0)) / df['l_at']
df['marginch_an']         = (df['ib'] / df['sale']) - (df['l_ib'] / df['l_sale'])
df['pchdeprn_an']         = ((df['dp'] / df['ppent']) - (df['l_dp'] / df['l_ppent'])) / (df['l_dp'] / df['l_ppent'])
df['pchgm2pchsale_an']    = (((df['sale'] - df['cogs']) - (df['l_sale'] - df['l_cogs'])) / (df['l_sale'] - df['l_cogs'])) - ((df['sale'] - df['l_sale']) / df['l_sale'])
df['pchquickratio_an']    = (((df['act'] - df['invt']) / df['lct'] - (df['l_act'] - df['l_invt']) / df['l_lct'])) / ((df['l_act'] - df['l_invt']) / df['l_lct'])
df['pchsale2pchinvt_an']  = ((df['sale'] - df['l_sale']) / df['l_sale']) - ((df['invt'] - df['l_invt']) / df['l_invt'])
df['pchsale2pchrect_an']  = ((df['sale'] - df['l_sale']) / df['l_sale']) - ((df['rect'] - df['l_rect']) / df['l_rect'])
df['pchsale2pchxsga_an']  = ((df['sale'] - df['l_sale']) / df['l_sale']) - ((df['xsga'] - df['l_xsga']) / df['l_xsga'])
df['pchsales2inv_an']     = ((df['sale'] / df['invt']) - (df['l_sale'] / df['l_invt'])) / (df['l_sale'] / df['l_invt'])

print("  ✓ Growth & Deltas complete")


Engineering ratios — (6) Growth & Deltas...
  ✓ Growth & Deltas complete


In [17]:
# Cell 16 — Fama-French 49 Industry Classification
# Source: Ken French Data Library — Siccodes49.txt (verbatim)
# https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_49_ind_port.html
# NOTE: Industry 36 (Softw — Computer Software) was MISSING from earlier implementations.
#       Without it, all Computer Software SICs fall into Business Services (34).
df = df.copy()
print("Assigning Fama-French 49 industry codes...")

FF49_RULES = [
    ( 1, 'Agric', 'Agriculture',                  [(100,199),(200,299),(700,799),(910,919),(2048,2048)]),
    ( 2, 'Food',  'Food Products',                [(2000,2009),(2010,2019),(2020,2029),(2030,2039),(2040,2046),(2050,2059),(2060,2063),(2070,2079),(2090,2092),(2095,2095),(2098,2099)]),
    ( 3, 'Soda',  'Candy & Soda',                 [(2064,2068),(2086,2086),(2087,2087),(2096,2096),(2097,2097)]),
    ( 4, 'Beer',  'Beer & Liquor',                [(2080,2080),(2082,2082),(2083,2083),(2084,2084),(2085,2085)]),
    ( 5, 'Smoke', 'Tobacco Products',             [(2100,2199)]),
    ( 6, 'Toys',  'Recreation',                   [(920,999),(3650,3651),(3652,3652),(3732,3732),(3930,3931),(3940,3949)]),
    ( 7, 'Fun',   'Entertainment',                [(7800,7829),(7830,7833),(7840,7841),(7900,7900),(7910,7911),(7920,7929),(7930,7933),(7940,7949),(7980,7980),(7990,7999)]),
    ( 8, 'Books', 'Printing and Publishing',      [(2700,2709),(2710,2719),(2720,2729),(2730,2739),(2740,2749),(2770,2771),(2780,2789),(2790,2799)]),
    ( 9, 'Hshld', 'Consumer Goods',               [(2047,2047),(2391,2392),(2510,2519),(2590,2599),(2840,2843),(2844,2844),(3160,3161),(3170,3172),(3190,3199),(3229,3229),(3260,3260),(3262,3263),(3269,3269),(3230,3231),(3630,3639),(3750,3751),(3800,3800),(3860,3861),(3870,3873),(3910,3911),(3914,3915),(3960,3962),(3991,3991),(3995,3995)]),
    (10, 'Clths', 'Apparel',                      [(2300,2390),(3020,3021),(3100,3111),(3130,3131),(3140,3149),(3150,3151),(3963,3965)]),
    (11, 'Hlth',  'Healthcare',                   [(8000,8099)]),
    (12, 'MedEq', 'Medical Equipment',            [(3693,3693),(3840,3849),(3850,3851)]),
    (13, 'Drugs', 'Pharmaceutical Products',      [(2830,2836)]),
    (14, 'Chems', 'Chemicals',                    [(2800,2809),(2810,2819),(2820,2829),(2850,2859),(2860,2869),(2870,2879),(2890,2899)]),
    (15, 'Rubbr', 'Rubber and Plastic Products',  [(3031,3031),(3041,3041),(3050,3053),(3060,3069),(3070,3079),(3080,3089),(3090,3099)]),
    (16, 'Txtls', 'Textiles',                     [(2200,2269),(2270,2279),(2280,2284),(2290,2295),(2297,2299),(2393,2395),(2397,2399)]),
    (17, 'BldMt', 'Construction Materials',       [(800,899),(2400,2439),(2450,2459),(2490,2499),(2660,2661),(2950,2952),(3200,3200),(3210,3211),(3240,3241),(3250,3259),(3261,3261),(3264,3264),(3270,3275),(3280,3281),(3290,3293),(3295,3299),(3420,3429),(3430,3433),(3440,3442),(3446,3446),(3448,3449),(3450,3452),(3490,3499),(3996,3996)]),
    (18, 'Cnstr', 'Construction',                 [(1500,1511),(1520,1529),(1530,1539),(1540,1549),(1600,1699),(1700,1799)]),
    (19, 'Steel', 'Steel Works Etc',              [(3300,3300),(3310,3317),(3320,3325),(3330,3339),(3340,3341),(3350,3357),(3360,3369),(3370,3379),(3390,3399)]),
    (20, 'FabPr', 'Fabricated Products',          [(3400,3400),(3443,3444),(3460,3469),(3470,3479)]),
    (21, 'Mach',  'Machinery',                    [(3510,3519),(3520,3529),(3530,3538),(3540,3549),(3550,3559),(3560,3569),(3580,3582),(3585,3586),(3589,3589),(3590,3599)]),
    (22, 'ElcEq', 'Electrical Equipment',         [(3600,3600),(3610,3613),(3620,3621),(3623,3629),(3640,3649),(3660,3660),(3690,3692),(3699,3699)]),
    (23, 'Autos', 'Automobiles and Trucks',       [(2296,2296),(2396,2396),(3010,3011),(3537,3537),(3647,3647),(3694,3694),(3700,3700),(3710,3716),(3790,3792),(3799,3799)]),
    (24, 'Aero',  'Aircraft',                     [(3720,3721),(3723,3725),(3728,3729)]),
    (25, 'Ships', 'Shipbuilding, Railroad Eq.',   [(3730,3731),(3740,3743)]),
    (26, 'Guns',  'Defense',                      [(3760,3769),(3795,3795),(3480,3489)]),
    (27, 'Gold',  'Precious Metals',              [(1040,1049)]),
    (28, 'Mines', 'Non-Metallic Mining',          [(1000,1039),(1050,1119),(1400,1499)]),
    (29, 'Coal',  'Coal',                         [(1200,1299)]),
    (30, 'Oil',   'Petroleum and Natural Gas',    [(1300,1339),(1370,1389),(2900,2912),(2990,2999)]),
    (31, 'Util',  'Utilities',                    [(4900,4911),(4920,4925),(4930,4932),(4939,4942)]),
    (32, 'Telcm', 'Communication',                [(4800,4800),(4810,4813),(4820,4822),(4830,4841),(4880,4892),(4899,4899)]),
    (33, 'PerSv', 'Personal Services',            [(7020,7021),(7030,7033),(7200,7219),(7220,7221),(7230,7241),(7250,7251),(7260,7299),(7395,7395),(7500,7500),(7510,7515),(7520,7549),(7600,7600),(7620,7631),(7640,7641),(7690,7699),(8100,8499),(8600,8699),(8800,8899)]),
    (34, 'BusSv', 'Business Services',            [(2750,2759),(3993,3993),(7218,7218),(7300,7300),(7310,7319),(7320,7342),(7349,7353),(7359,7360),(7369,7369),(7374,7374),(7376,7380),(7381,7385),(7389,7397),(7399,7399),(7519,7519),(8700,8748),(8900,8999),(4220,4229)]),
    (35, 'Hardw', 'Computers',                    [(3570,3579),(3680,3689),(3695,3695)]),
    (36, 'Softw', 'Computer Software',            [(7370,7372),(7373,7373),(7375,7375)]),   # ← was MISSING
    (37, 'Chips', 'Electronic Equipment',         [(3622,3622),(3661,3669),(3670,3679),(3810,3810),(3812,3812)]),
    (38, 'LabEq', 'Measuring and Control Eq.',    [(3811,3811),(3820,3829),(3830,3839)]),
    (39, 'Paper', 'Business Supplies',            [(2520,2549),(2600,2639),(2670,2699),(2760,2761),(3950,3955)]),
    (40, 'Boxes', 'Shipping Containers',          [(2440,2449),(2640,2659),(3220,3221),(3410,3412)]),
    (41, 'Trans', 'Transportation',               [(4000,4013),(4040,4049),(4100,4199),(4200,4219),(4230,4231),(4400,4499),(4500,4599),(4600,4699),(4700,4712),(4720,4749),(4780,4785),(4789,4789)]),
    (42, 'Whlsl', 'Wholesale',                    [(5000,5000),(5010,5015),(5020,5023),(5030,5049),(5050,5059),(5060,5065),(5070,5078),(5080,5092),(5093,5094),(5099,5099),(5100,5100),(5110,5113),(5120,5122),(5130,5199)]),
    (43, 'Rtail', 'Retail',                       [(5200,5271),(5300,5399),(5400,5499),(5500,5599),(5600,5699),(5700,5736),(5750,5799),(5900,5912),(5920,5932),(5940,5999)]),
    (44, 'Meals', 'Restaurants, Hotels, Motels',  [(5800,5829),(5890,5899),(7000,7019),(7040,7049),(7213,7213)]),
    (45, 'Banks', 'Banking',                      [(6000,6062),(6080,6082),(6090,6199)]),
    (46, 'Insur', 'Insurance',                    [(6300,6331),(6350,6361),(6370,6379),(6390,6411)]),
    (47, 'RlEst', 'Real Estate',                  [(6500,6515),(6517,6532),(6540,6541),(6550,6553),(6590,6599),(6610,6611)]),
    (48, 'Fin',   'Trading',                      [(6200,6299),(6700,6726),(6730,6733),(6740,6799)]),
    (49, 'Other', 'Almost Nothing',               [(4950,4959),(4960,4961),(4970,4971),(4990,4991)]),
]

_ff49_lookup = {}
for _num, _abbr, _name, _ranges in FF49_RULES:
    for _lo, _hi in _ranges:
        for _sic in range(_lo, _hi + 1):
            if _sic not in _ff49_lookup:
                _ff49_lookup[_sic] = (_num, _abbr, _name)

def assign_ff49(sic):
    if pd.isna(sic):
        return 49, 'Other', 'Almost Nothing'
    return _ff49_lookup.get(int(float(sic)), (49, 'Other', 'Almost Nothing'))

_ff49 = df['sic'].apply(assign_ff49)
df['ff49_num']  = _ff49.apply(lambda x: x[0])
df['ff49_abbr'] = _ff49.apply(lambda x: x[1])
df['industry']  = _ff49.apply(lambda x: x[2])

print(f"FF49 mapping complete: {df['industry'].nunique()} industries assigned")
print(f"  Softw firms: {(df['ff49_num']==36).sum():,} firm-years (sanity check — should be >0)")
print(f"  Other (49):  {(df['ff49_num']==49).sum():,} firm-years")


Assigning Fama-French 49 industry codes...
FF49 mapping complete: 49 industries assigned
  Softw firms: 8,336 firm-years (sanity check — should be >0)
  Other (49):  2,639 firm-years


In [18]:
# Cell 17 — aliases, infinity cleanup, EV construction, sample filters
# Final aliases
df = df.copy()
df['assets']   = df['at']
df['book']     = df['be']
df['sale_ac']  = df['sale']
df['CAPMbeta'] = np.nan   # Requires CRSP monthly returns; set in a separate step

# Replace infinities introduced by ratio divisions
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Enterprise value
df['market_cap'] = df['csho'] * df['prcc_f']
df['ev'] = df['market_cap'] + df['dltt'].fillna(0) + df['dlc'].fillna(0) - df['che'].fillna(0)

# --- Filters (config.py: MIN_PRICE=1.0, MIN_MKTCAP=50.0) ---
n_before = len(df)
df = df[(df['prcc_f'] >= MIN_PRICE) & (df['market_cap'] >= MIN_MKTCAP)]
_log.append(('4. After penny/micro-cap filter', len(df), df['gvkey'].nunique()))
print(f"Penny/micro-cap filter: dropped {n_before - len(df):,} firm-years")

n_before = len(df)
df = df[(df['ev'] > 0) & (df['seq'] > 0) & (df['at'] > 0) & (df['sale'] > 0)]
_log.append(('5. After quality filter (ev/seq/at/sale > 0)', len(df), df['gvkey'].nunique()))
print(f"Quality filter: dropped {n_before - len(df):,} firm-years")

# --- Restrict to study years (config.py: YEARS = [2020..2024]) ---
n_before = len(df)
df = df[df['fyear'].isin(YEARS)]
_log.append((f'6. After fyear filter ({min(YEARS)}-{max(YEARS)})', len(df), df['gvkey'].nunique()))
print(f"Year filter {YEARS}: dropped {n_before - len(df):,} firm-years outside sample window")


print(f"\nFinal sample: {len(df):,} firm-years | {df['gvkey'].nunique():,} unique firms")


Penny/micro-cap filter: dropped 43,435 firm-years
Quality filter: dropped 29,831 firm-years
Year filter [2020, 2021, 2022, 2023, 2024]: dropped 33,799 firm-years outside sample window

Final sample: 20,883 firm-years | 5,700 unique firms


In [19]:
# Cell 18 — pipeline attrition summary
print("=" * 65)
print("SAMPLE ATTRITION — DATA PREPARATION PIPELINE")
print("=" * 65)

attrition = pd.DataFrame(_log, columns=['Step', 'Firm-Years', 'Companies'])
attrition['FY Dropped'] = attrition['Firm-Years'].shift(1) - attrition['Firm-Years']
attrition['Co Dropped'] = attrition['Companies'].shift(1) - attrition['Companies']
attrition['FY Dropped'] = attrition['FY Dropped'].fillna(0).astype(int)
attrition['Co Dropped'] = attrition['Co Dropped'].fillna(0).astype(int)
attrition['% FY Kept']  = (attrition['Firm-Years'] / attrition['Firm-Years'].iloc[0] * 100).round(1)

print(attrition.to_string(index=False))


SAMPLE ATTRITION — DATA PREPARATION PIPELINE
                                        Step  Firm-Years  Companies  FY Dropped  Co Dropped  % FY Kept
                                 1. Raw file      169724      21674           0           0      100.0
                         2. After USD filter      141563      18111       28161        3563       83.4
                      3. After deduplication      127948      18111       13615           0       75.4
             4. After penny/micro-cap filter       84513      12681       43435        5430       49.8
5. After quality filter (ev/seq/at/sale > 0)       54682       8063       29831        4618       32.2
           6. After fyear filter (2020-2024)       20883       5700       33799        2363       12.3


In [20]:
# Cell 19 — FF49 industry coverage summary
summary = (
    df.groupby(['ff49_num', 'ff49_abbr', 'industry'])
    .agg(firm_years=('gvkey', 'count'), companies=('gvkey', 'nunique'))
    .reset_index()
    .sort_values('ff49_num')
    .rename(columns={'ff49_num': 'FF49', 'ff49_abbr': 'Abbr',
                     'industry': 'Industry', 'firm_years': 'Firm-Years', 'companies': 'Companies'})
)
summary['% of Total'] = (summary['Firm-Years'] / summary['Firm-Years'].sum() * 100).round(1)

totals = pd.DataFrame([{
    'FF49': '', 'Abbr': '', 'Industry': 'TOTAL',
    'Firm-Years': summary['Firm-Years'].sum(),
    'Companies': df['gvkey'].nunique(),
    '% of Total': 100.0
}])
summary = pd.concat([summary, totals], ignore_index=True)
print(summary.to_string(index=False))


FF49  Abbr                    Industry  Firm-Years  Companies  % of Total
   1 Agric                 Agriculture          56         15         0.3
   2  Food               Food Products         290         72         1.4
   3  Soda                Candy & Soda          64         14         0.3
   4  Beer               Beer & Liquor          84         20         0.4
   5 Smoke            Tobacco Products          20          6         0.1
   6  Toys                  Recreation         114         29         0.5
   7   Fun               Entertainment         203         62         1.0
   8 Books     Printing and Publishing          53         13         0.3
   9 Hshld              Consumer Goods         255         65         1.2
  10 Clths                     Apparel         186         43         0.9
  11  Hlth                  Healthcare         308         95         1.5
  12 MedEq           Medical Equipment         621        179         3.0
  13 Drugs     Pharmaceutical Products

In [21]:
# Cell 20 — save panel_clean.parquet
PANEL_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(PANEL_CLEAN, index=False)
print(f"Saved: {PANEL_CLEAN}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Years: {sorted(df['fyear'].unique())}")
print(f"\nColumn count by group:")
print(f"  Compustat raw items  : {sum(1 for c in df.columns if c in ['sale','at','seq','lt','ni','ebitda','ebit','oancf','capx','dv','che','rect','invt','ppent','lct','act','ap','ceq','pstk','mib','icapt','txditc','cogs','xsga','xrd','dp','xint','spi','nopi','pi','txt','mii','ib','ibcom'])}")
print(f"  Lag / delta cols     : {sum(1 for c in df.columns if c.startswith('l_') or c.startswith('d_'))}")
ratio_cols = [c for c in df.columns if c.endswith('_wr') or c.endswith('_an')]
print(f"  Engineered ratios    : {len(ratio_cols)}")
print(f"  Identifiers + other  : remaining")



Saved: /work/Repo/notebooks/../data/processed/panel_clean.parquet
Shape: 20,883 rows × 206 columns
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Column count by group:
  Compustat raw items  : 34
  Lag / delta cols     : 52
  Engineered ratios    : 91
  Identifiers + other  : remaining
